# Backpropagation
**Machine Learning · Week 5 (Neural networks with PyTorch) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/ml/w5d3.html

Backpropagation: compute the network forward, then push the gradient of the loss backward through
every step with the chain rule. We do it by hand once, then let autograd do it.

In [ ]:
import math
import torch

torch.set_num_threads(1)
x, y = 1.0, 1.0                                   # one example
w1, b1, w2, b2 = 1.5, -0.5, -2.0, 0.5            # four knobs

z1 = w1 * x + b1
h = math.tanh(z1)
z2 = w2 * h + b2
p = 1 / (1 + math.exp(-z2))
L = -(y * math.log(p) + (1 - y) * math.log(1 - p))
print(f"forward:  z1 = {z1:.4f}, h = {h:.4f}, z2 = {z2:.4f}, p = {p:.4f}, loss = {L:.4f}")

## Backward, by hand
Start at the loss and walk back. Each step multiplies by one local slope (the chain rule).

In [ ]:
dz2 = p - y                       # sigmoid + cross-entropy: the famous p - y
dw2, db2 = dz2 * h, dz2           # z2 = w2*h + b2
dh = dz2 * w2
dz1 = dh * (1 - h ** 2)           # d tanh(z)/dz = 1 - tanh(z)^2
dw1, db1 = dz1 * x, dz1
print(f"by hand:  dL/dw1 = {dw1:.4f}, dL/db1 = {db1:.4f}, dL/dw2 = {dw2:.4f}, dL/db2 = {db2:.4f}")

## The same with autograd
PyTorch records the graph during the forward pass and runs exactly these steps in `backward()`.

In [ ]:
W1, B1, W2, B2 = (torch.tensor(v, requires_grad=True) for v in (w1, b1, w2, b2))
P = torch.sigmoid(W2 * torch.tanh(W1 * x + B1) + B2)
loss = -(y * torch.log(P) + (1 - y) * torch.log(1 - P))
loss.backward()
print(f"autograd: dL/dw1 = {W1.grad:.4f}, dL/db1 = {B1.grad:.4f}, dL/dw2 = {W2.grad:.4f}, dL/db2 = {B2.grad:.4f}")

## Why backward? Count the work
A finite-difference gradient needs one extra forward pass **per parameter**. Backpropagation gets all of them
from one backward pass, which costs about as much as one forward pass.

In [ ]:
import time

net = torch.nn.Sequential(torch.nn.Linear(256, 256), torch.nn.Tanh(), torch.nn.Linear(256, 1))
n_params = sum(q.numel() for q in net.parameters())
xb = torch.randn(64, 256)
t0 = time.perf_counter()
out = net(xb).pow(2).mean()
t_forward = time.perf_counter() - t0
t0 = time.perf_counter()
out.backward()
t_backward = time.perf_counter() - t0
print(f"{n_params:,} parameters")
print(f"one forward pass:  {t_forward * 1000:.2f} ms")
print(f"one backward pass: {t_backward * 1000:.2f} ms  -> every gradient at once")
fd = n_params * t_forward
print(f"finite differences would need {n_params:,} forward passes: about {fd:.0f} seconds instead of {t_backward * 1000:.1f} ms")
print("and the gap grows with every parameter: for a network with a billion weights it's the difference between possible and not.")

## Your turn: a gradient check
Write `numeric_grad`: nudge one parameter by ±ε, recompute the loss, and use
$\frac{L(w + \varepsilon) - L(w - \varepsilon)}{2\varepsilon}$. Compare with the backprop values above.
(Professionals use exactly this check when they write new layers.)

In [ ]:
def loss_of(w1, b1, w2, b2):
    p = 1 / (1 + math.exp(-(w2 * math.tanh(w1 * x + b1) + b2)))
    return -(y * math.log(p) + (1 - y) * math.log(1 - p))

def numeric_grad(params, i, eps=1e-6):
    return 0.0     # TODO: copy params twice, add and subtract eps at position i, apply the formula

params = [w1, b1, w2, b2]
for i, name in enumerate(["w1", "b1", "w2", "b2"]):
    print(f"dL/d{name}: numeric {numeric_grad(params, i):.6f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def numeric_grad(params, i, eps=1e-6):
    up, down = list(params), list(params)
    up[i] += eps
    down[i] -= eps
    return (loss_of(*up) - loss_of(*down)) / (2 * eps)

for i, (name, exact) in enumerate(zip(["w1", "b1", "w2", "b2"], [dw1, db1, dw2, db2])):
    print(f"dL/d{name}: numeric {numeric_grad(params, i):.6f}   backprop {exact:.6f}")